# 01 EDA - Full Raw Dataset

This notebook scans every JSONL file under `data/raw/` in chunks. It does not load the full 2M-row raw dataset into memory at once.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
from src.full_pipeline import RAW_DIR, EXPERIMENTS_DIR, discover_raw_files, summarize_raw_jsonl, write_json

sns.set_theme(style="whitegrid")
%matplotlib inline


## 1. Stream Raw Files


In [ ]:
RAW_CHUNKSIZE = 100_000

raw_files = discover_raw_files(RAW_DIR)
print("Raw files:")
for path in raw_files:
    print(f"- {path.name}")

summary = summarize_raw_jsonl(RAW_DIR, chunksize=RAW_CHUNKSIZE)
write_json(EXPERIMENTS_DIR / 'raw_eda_summary.json', summary)

print(f"Total rows: {summary['total_rows']:,}")
print("Rows by category:")
for category, count in summary['category_counts'].items():
    print(f"- {category}: {count:,}")


## 2. Rating And Category Distributions


In [ ]:
rating_df = pd.DataFrame({
    'rating': [float(k) for k in summary['rating_counts'].keys()],
    'count': list(summary['rating_counts'].values()),
}).sort_values('rating')

category_df = pd.DataFrame({
    'category': list(summary['category_counts'].keys()),
    'count': list(summary['category_counts'].values()),
}).sort_values('category')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=rating_df, x='rating', y='count', ax=axes[0], palette='viridis')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

sns.barplot(data=category_df, x='category', y='count', ax=axes[1], palette='Set2')
axes[1].set_title('Rows Per Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 3. Text Length Summary


In [ ]:
print("Review Length Statistics (Characters):")
display(pd.Series(summary['text_length_describe']).to_frame('text_length'))

print("Word Count Statistics:")
display(pd.Series(summary['word_count_describe']).to_frame('word_count'))

verified = summary['verified_purchase_percent']
if verified is not None:
    print(f"Percentage of verified purchases: {verified:.1f}%")
